# Wallet Strategy Selection

Unified selection stage: runs **all wallet-selection methods** and saves each
as a named `WalletSelectionStrategy` artifact in the workspace.  The backtest
stage loads these artifacts directly — no strategy logic is re-built there.

## Methods included
| id suffix | description |
|-----------|-------------|
| `skill_sweep` → `quality_core` | top-N by best skill metric (grid-searched on train-a→train-b) |
| `cohort_early_entry` | wallets that enter markets early |
| `cohort_smooth_pnl` | wallets with high PnL relative to volatility |
| `volatility` | volatility-filtered wallets (from the profitable_wallet_analysis path) |

Each method produces **two trigger variants**:
- `score_threshold` — composite signal score ≥ calibrated threshold (Kelly sizing)
- `all_open_buys`   — every open-buy event (fixed stake baseline)


In [480]:

%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [481]:
# ── configuration ─────────────────────────────────────────────────────────────
RECENCY_DAYS     = 90

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

METRICS_A_PATH          = WORKSPACE_DIR / "wallet_metrics_train_a.parquet"
METRICS_B_PATH          = WORKSPACE_DIR / "wallet_metrics_train_b.parquet"
METRICS_FULL_PATH       = WORKSPACE_DIR / "wallet_metrics_full_train.parquet"
METRICS_TEST_PATH       = WORKSPACE_DIR / "wallet_metrics_test.parquet"
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH       = WORKSPACE_DIR / "signal_events_test.parquet"


## Derive train/test dates from data

In [482]:
# ── derive train/test boundary from is_train flag ───────────────────────────
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
# Split train_a / train_b at the trade-count median of training rows so that
# both halves always contain roughly equal numbers of trades even when the
# data is unevenly distributed over calendar time.
_train_rows = _sample.loc[_sample["is_train"]].sort_values("dt")
TRAIN_A_END_DATE = _train_rows["dt"].quantile(0.5, interpolation="nearest").date()
del _sample, _train_rows
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-01-03  TRAIN_A_END_DATE=2026-01-02


In [483]:
trades_df = dataset.to_table().to_pandas()

In [484]:
print(len(trades_df))
trades_df.head(3)

389972


,wallet,condition_id,dt,side,outcome,position,total_quantity,avg_price,trade_value_usdc,final_value_usdc,trade_pnl,token_winner,final_price,tx_hash,num_fills,is_train
0,0x00cefdf8f1902d360230b2a5c85e72f7016c3f01,0x0ac2ffe56e0baf504292ab168e3194febb6d0904905817a0950923cbf1e30a79,2026-01-05 02:26:33+00:00,BUY,Suns,1000.000000,1000.000000,0.150,150.000000,1000.0,850.000000,True,1.0,0x86d7a0cec149bd124f6f69fcdc2e3ef25377d055daf44cbde8962a5c7aa5e92d,8,False
1,0x00cefdf8f1902d360230b2a5c85e72f7016c3f01,0x0ac2ffe56e0baf504292ab168e3194febb6d0904905817a0950923cbf1e30a79,2026-01-05 04:17:39+00:00,SELL,Suns,0.000000,1000.000000,0.999,999.000000,1000.0,-1.000000,True,1.0,0x1f661275d8bc3627f2ea9dd835f2173dcff3d109c3846d1db146a8a31bf36703,1,False
2,0x014c23bfc4f0f5771d757b34a24435cc2466f5b6,0x0d063d604084788d0a0a87c7f1048853ddd0d147e36179aec8ec61271f9e9b04,2026-01-04 17:57:49+00:00,BUY,Titans,3061.224488,3061.224488,0.490,1499.999999,0.0,-1499.999999,False,0.0,0x55fffd757796db8dcfee6e710b9953a2a5615213fa6ae3c7ba9b25890582e3d3,1,False


In [485]:
len(trades_df.groupby(["tx_hash", "wallet", 'side']))

389972

## Compute / load wallet skill metrics

In [486]:
from polymarket_analysis.wallet_selection.metrics import compute_wallet_skill_workspace

if all(p.exists() for p in [METRICS_A_PATH, METRICS_B_PATH, METRICS_FULL_PATH, METRICS_TEST_PATH]):
    train_a_metrics    = pd.read_parquet(METRICS_A_PATH)
    train_b_metrics    = pd.read_parquet(METRICS_B_PATH)
    full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
    test_metrics       = pd.read_parquet(METRICS_TEST_PATH)
else:
    train_a_metrics, train_b_metrics, full_train_metrics, test_metrics = (
        compute_wallet_skill_workspace(
            dataset,
            end_date_train=END_DATE_TRAIN,
            train_a_end_date=TRAIN_A_END_DATE,
            recency_days=RECENCY_DAYS,
        )
    )
    train_a_metrics.to_parquet(METRICS_A_PATH, index=False)
    train_b_metrics.to_parquet(METRICS_B_PATH, index=False)
    full_train_metrics.to_parquet(METRICS_FULL_PATH, index=False)
    test_metrics.to_parquet(METRICS_TEST_PATH, index=False)

pd.Series({
    "train_a_wallets":    len(train_a_metrics),
    "train_b_wallets":    len(train_b_metrics),
    "full_train_wallets": len(full_train_metrics),
    "test_wallets":       len(test_metrics),
}).to_frame("value")


,value
train_a_wallets,2372
train_b_wallets,2372
full_train_wallets,2372
test_wallets,2372


## Cohort selection sweep (skill path)

Grid-search over metrics × top-N using train-a → train-b persistence.

In [487]:
from polymarket_analysis.wallet_selection.selector import (
    CANDIDATE_METRICS,
    cohort_selection_sweep,
    select_wallets,
)

cohort_sweep = cohort_selection_sweep(train_a_metrics, train_b_metrics, CANDIDATE_METRICS)
cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge"], ascending=False
).head(20)


,metric,top_n,wallets_found_in_train_b,train_b_open_buy_trades,train_b_weighted_prob_edge,train_b_avg_prob_edge,train_b_avg_copy_roi_capped,train_b_total_copy_pnl_usdc,train_b_hit_rate
20,roi_sharpe,50,50,357,0.347712,0.200713,0.405138,6.903763e+05,0.694678
15,edge_sharpe,50,50,383,0.352715,0.175398,0.385438,6.472729e+05,0.704961
10,avg_copy_roi_capped,50,50,412,0.337054,0.167781,0.373972,8.208891e+05,0.635922
0,prob_edge_shrunk,50,50,524,0.330611,0.163133,0.346930,9.419363e+05,0.669847
11,avg_copy_roi_capped,100,100,1147,0.264458,0.113893,0.326760,1.124658e+06,0.591107
21,roi_sharpe,100,100,1166,0.263413,0.116026,0.326626,1.123054e+06,0.597770
16,edge_sharpe,100,100,1139,0.266537,0.116402,0.299261,1.128115e+06,0.614574
30,hit_rate,50,50,384,0.274846,0.159371,0.296260,5.926818e+05,0.747396
31,hit_rate,100,100,1031,0.251522,0.119720,0.281982,1.144875e+06,0.655674
1,prob_edge_shrunk,100,100,1843,0.260376,0.084464,0.236349,1.209256e+06,0.572979


In [488]:
# pick best metric / top-N
best_row = cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge", "train_b_open_buy_trades"],
    ascending=[False, False, False],
).iloc[0]
BEST_SELECTION_METRIC = best_row["metric"]
BEST_TOP_N            = int(best_row["top_n"])
{"best_metric": BEST_SELECTION_METRIC, "best_top_n": BEST_TOP_N}


{'best_metric': 'roi_sharpe', 'best_top_n': 50}

## Select wallets (skill path) + build cohorts

In [489]:
selected_wallets = select_wallets(full_train_metrics, BEST_SELECTION_METRIC, BEST_TOP_N)
selected_wallets.to_parquet(WORKSPACE_DIR / "selected_wallets_v2.parquet", index=False)
selected_wallets[[
    "wallet", "open_buy_trades", "distinct_markets",
    "recent_open_buy_trades", BEST_SELECTION_METRIC, "wallet_quality"
]].head(15)


,wallet,open_buy_trades,distinct_markets,recent_open_buy_trades,roi_sharpe,wallet_quality
0,0x84cb17a50bc2487e8d64029783c3d2abcba328ad,37,36,37,0.780220,1.00
1,0x6bb33bdb14cea8f53766539efc6019b55e8a758e,25,25,25,0.694642,0.98
2,0x03524d9d00cffe5004d1d270edfea7b3109e3292,23,21,23,0.647861,0.96
3,0xcc6385889a3466743f0ce6bd46a2b20b86d0f1c7,38,31,38,0.588241,0.94
4,0xd39701235dcf439c490b12bd4f546e9d7af95052,33,32,33,0.587330,0.92
5,0xd7b6ff712db56bee267921da6f4a613db03431e9,24,24,24,0.563305,0.90
6,0xc257ea7e3a81ca8e16df8935d44d513959fa358e,21,21,21,0.558939,0.88
7,0x9fad8308ef6b6ed5320e53a290a3bd4ad91f5a9f,43,42,43,0.550871,0.86
8,0x448861155279dbf833d041b963e3ac854599e319,29,28,29,0.542038,0.84
9,0x439b4cac193239c5769ac26e1d0b167e22182c9a,20,20,20,0.534486,0.82


## Build wallet profiles and signal events

In [490]:
from polymarket_analysis.signal.builder import (
    build_wallet_profiles,
    build_signal_events,
)

selected_wallet_profiles = build_wallet_profiles(
    dataset, selected_wallets, period="full_train",
    end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
)
selected_wallet_profiles.to_parquet(
    WORKSPACE_DIR / "selected_wallet_profiles_v2.parquet", index=False
)

if CALIBRATION_SIGNALS_PATH.exists():
    train_b_open_buys = pd.read_parquet(CALIBRATION_SIGNALS_PATH)
else:
    _, train_b_open_buys = build_signal_events(
        dataset, selected_wallet_profiles, period="train_b",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
        price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
    )
    train_b_open_buys.to_parquet(CALIBRATION_SIGNALS_PATH, index=False)

if TEST_SIGNALS_PATH.exists():
    test_open_buys = pd.read_parquet(TEST_SIGNALS_PATH)
else:
    _, test_open_buys = build_signal_events(
        dataset, selected_wallet_profiles, period="test",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
        price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
    )
    test_open_buys.to_parquet(TEST_SIGNALS_PATH, index=False)

print(f"train_b signals: {len(train_b_open_buys):,}  test signals: {len(test_open_buys):,}")


train_b signals: 697  test signals: 807


## Calibrate signal scoring on train-B

In [491]:
from polymarket_analysis.signal.scorer import (
    build_calibration_tables,
    apply_signal_score,
    select_signal_threshold,
)

price_table, consensus_table, global_edge = build_calibration_tables(train_b_open_buys)
train_b_scored = apply_signal_score(train_b_open_buys, price_table, consensus_table)
SIGNAL_THRESHOLD = select_signal_threshold(train_b_scored)
print(f"Global edge: {global_edge:.4f}")
print(f"Selected signal threshold: {SIGNAL_THRESHOLD:.2f}")

# score distribution
score_deciles = (
    train_b_scored.assign(
        score_decile=lambda df: pd.qcut(df["signal_score"], q=10, labels=False, duplicates="drop")
    )
    .groupby("score_decile")
    .agg(
        count=("signal_score", "size"),
        avg_signal_score=("signal_score", "mean"),
        avg_copy_roi_capped=("copy_roi_capped", "mean"),
    )
    .reset_index()
)
score_deciles


Global edge: 0.1803
Selected signal threshold: 0.75


,score_decile,count,avg_signal_score,avg_copy_roi_capped
0,0,70,0.254320,0.334564
1,1,70,0.345275,0.018908
2,2,69,0.394273,0.267481
3,3,71,0.440305,0.358682
4,4,70,0.480850,0.648767
5,5,68,0.529984,0.366804
6,6,70,0.583944,0.591862
7,7,69,0.635727,0.879987
8,8,70,0.684268,0.451402
9,9,70,0.771784,0.746221


## Build wallet cohorts (skill path)

In [492]:
from polymarket_analysis.wallet_selection.selector import build_wallet_cohorts

wallet_cohorts = build_wallet_cohorts(
    full_train_metrics, train_b_open_buys, selected_wallets,
    top_n=BEST_TOP_N,
)
{name: len(df) for name, df in wallet_cohorts.items()}


{'quality_core': 50, 'early_entry': 48, 'smooth_pnl': 50}

## Volatility-based wallet selection (second method)

Loads the full training set, applies the volatility filter, and stores the
result as an additional `WalletSelectionStrategy`.  The volatility wallet set
is added to `wallet_cohorts` so the strategy factory handles it uniformly.

In [493]:
from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)

# Load full training trades for the volatility path
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_train_vol = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)
df_train_vol["dt"] = pd.to_datetime(df_train_vol["dt"], utc=True)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_train_vol.columns and "quantity" not in df_train_vol.columns:
    df_train_vol = df_train_vol.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_train_vol["usdc_amount"]      = df_train_vol["usdc_amount"].astype(float)
df_train_vol["final_value_usdc"] = df_train_vol["final_value_usdc"].astype(float)
df_train_vol["quantity"]         = df_train_vol["quantity"].astype(float)

# PnL and notional per fill
df_train_vol["pnl"] = np.where(
    df_train_vol["side"] == "BUY",
    df_train_vol["final_value_usdc"] - df_train_vol["usdc_amount"],
    df_train_vol["usdc_amount"] - df_train_vol["final_value_usdc"],
)
df_train_vol["notional"] = np.where(
    df_train_vol["side"] == "BUY",
    df_train_vol["usdc_amount"],
    df_train_vol["quantity"] * (1 - df_train_vol["price"].astype(float)),
)
df_slice = df_train_vol[df_train_vol["is_train"]].copy()

wallet_vol_train, _ = compute_wallet_metrics(df_slice)
del df_train_vol, df_slice

In [494]:
filtered_wallets_vol = filter_wallets_by_volatility(
    wallet_vol_train,
    min_buckets=20,
    max_top5_pnl_pct=1,
    max_top_market_pnl_pct=0.5,
)

filtered_wallets_vol = filtered_wallets_vol[filtered_wallets_vol['average_roi'] >= 0.04].sort_values("pnl_volatility", ascending=True).head(100)

print(f"Volatility-filtered wallets: {len(filtered_wallets_vol)}")

# Build wallet_quality based on total_pnl rank (higher = better)
filtered_wallets_vol = filtered_wallets_vol.copy().reset_index(drop=True)
filtered_wallets_vol["wallet_quality"] = filtered_wallets_vol["total_pnl"].rank(
    method="first", pct=True
)

# Add as additional cohort (uses only train-b signals for trigger calibration)
# We intersect with wallets that have signals to ensure coverage
vol_wallets_with_signals = set(train_b_open_buys["wallet"]).intersection(
    set(filtered_wallets_vol["wallet"])
)
wallet_cohorts["volatility"] = filtered_wallets_vol.copy().reset_index(drop=True)

# [
#     filtered_wallets_vol["wallet"].isin(vol_wallets_with_signals)
# ][["wallet", "wallet_quality"]].copy().reset_index(drop=True)

print(f"Volatility cohort: {len(wallet_cohorts['volatility'])}")

Volatility-filtered wallets: 55
Volatility cohort: 55


## Build and save strategy registry

All cohorts × all trigger variants → persisted `WalletSelectionStrategy` files.
The backtest loads these directly.

In [495]:
from polymarket_analysis.wallet_selection.selector import build_strategies_from_sweep
from polymarket_analysis.strategy.registry import save_strategy

all_strategies = build_strategies_from_sweep(
    wallet_cohorts=wallet_cohorts,
    signal_threshold=SIGNAL_THRESHOLD,
    selection_metric=BEST_SELECTION_METRIC,
    top_n=BEST_TOP_N,
    sweep_df=cohort_sweep,
    extra_metadata={
        "end_date_train": str(END_DATE_TRAIN),
        "train_a_end_date": str(TRAIN_A_END_DATE),
    },
)

for strategy in all_strategies:
    parquet_path, json_path = save_strategy(strategy, WORKSPACE_DIR)
    print(f"Saved [{strategy.strategy_id}]  wallets={len(strategy.wallets):3d}  trigger={strategy.trigger.fn_ref.split('.')[-1]}")

print(f"\nTotal strategies saved: {len(all_strategies)}")


Saved [quality_core__score_threshold]  wallets= 50  trigger=score_threshold
Saved [quality_core__all_open_buys]  wallets= 50  trigger=all_open_buys
Saved [early_entry__score_threshold]  wallets= 48  trigger=score_threshold
Saved [early_entry__all_open_buys]  wallets= 48  trigger=all_open_buys
Saved [smooth_pnl__score_threshold]  wallets= 50  trigger=score_threshold
Saved [smooth_pnl__all_open_buys]  wallets= 50  trigger=all_open_buys
Saved [volatility__score_threshold]  wallets= 55  trigger=score_threshold
Saved [volatility__all_open_buys]  wallets= 55  trigger=all_open_buys

Total strategies saved: 8


## Strategy registry summary

In [496]:
from polymarket_analysis.strategy.registry import load_all_strategies

registry = load_all_strategies(WORKSPACE_DIR)
summary_rows = []
for sid, s in registry.items():
    summary_rows.append({
        "strategy_id": s.strategy_id,
        "name": s.name,
        "selection_method": s.selection_method,
        "num_wallets": len(s.wallets),
        "trigger_fn": s.trigger.fn_ref.split(".")[-1],
        "threshold": s.trigger.params.get("threshold"),
        "dynamic_sizing": s.trigger.params.get("dynamic_sizing"),
    })

pd.DataFrame(summary_rows)


,strategy_id,name,selection_method,num_wallets,trigger_fn,threshold,dynamic_sizing
0,early_entry__all_open_buys,early_entry | all open-buys (fixed stake),cohort_early_entry,48,all_open_buys,NaN,False
1,early_entry__score_threshold,early_entry | score >= 0.75 (Kelly),cohort_early_entry,48,score_threshold,0.75,True
2,quality_core__all_open_buys,quality_core | all open-buys (fixed stake),skill_sweep,50,all_open_buys,NaN,False
3,quality_core__score_threshold,quality_core | score >= 0.75 (Kelly),skill_sweep,50,score_threshold,0.75,True
4,smooth_pnl__all_open_buys,smooth_pnl | all open-buys (fixed stake),cohort_smooth_pnl,50,all_open_buys,NaN,False
5,smooth_pnl__score_threshold,smooth_pnl | score >= 0.75 (Kelly),cohort_smooth_pnl,50,score_threshold,0.75,True
6,volatility__all_open_buys,volatility | all open-buys (fixed stake),volatility,55,all_open_buys,NaN,False
7,volatility__score_threshold,volatility | score >= 0.75 (Kelly),volatility,55,score_threshold,0.75,True


## Wallet PnL over time

Three independent plots:
- **Training plot** — cohort aggregate cumulative PnL over the training period only
- **Test plot** — cohort aggregate cumulative PnL over the test period only (starts from 0)
- **Individual plot** — per-wallet cumulative PnL spanning train + test, with a train/test split
  vline and wallet address labels; test portion resets to 0 at the split boundary

Set `PLOT_WALLET_PNL = False` to skip (e.g. when running headlessly).


In [497]:
PLOT_WALLET_PNL  = True
TOP_N_INDIVIDUAL = 10   # wallets shown in panel 1 per cohort


In [498]:
_trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_fills = pd.concat([pd.read_parquet(f) for f in _trade_files], ignore_index=True)
df_fills["dt"]               = pd.to_datetime(df_fills["dt"], utc=True)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
    df_fills = df_fills.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)

In [499]:
len(wallet_cohorts['volatility'])

55

In [500]:
vol_fills = df_fills[
    (df_fills["wallet"].isin(wallet_cohorts["volatility"]["wallet"]))
    & (df_fills["is_train"] == False)
].sort_values("dt")
len(vol_fills)

44916

In [501]:
from polymarket_analysis.visualization.wallet_plots import (
    plot_wallet_selection_pnl,
    plot_wallet_individual_pnl,
)

if PLOT_WALLET_PNL:
    _trade_files = sorted(TRADES_DIR.glob("*.parquet"))
    df_fills = pd.concat([pd.read_parquet(f) for f in _trade_files], ignore_index=True)
    df_fills["dt"]               = pd.to_datetime(df_fills["dt"], utc=True)

    # Normalise grouped schema → canonical fill-level names
    if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
        df_fills = df_fills.rename(columns={
            "total_quantity": "quantity",
            "avg_price": "price",
            "trade_value_usdc": "usdc_amount",
        })

    df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
    df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
    df_fills["quantity"]         = df_fills["quantity"].astype(float)

    split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

    # Cohort aggregate PnL — training period
    fig_train = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="train",
        title="Wallet selection cohorts — cohort cumulative PnL (training period)",
    )
    fig_train.show(renderer="browser")

    # Cohort aggregate PnL — test period (starts from 0)
    fig_test = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="test",
        title="Wallet selection cohorts — cohort cumulative PnL (test period)",
    )
    fig_test.show(renderer="browser")

    # # Individual wallet lines (train + test) with split vline and labels
    # fig_ind = plot_wallet_individual_pnl(
    #     df_fills,
    #     wallet_cohorts,
    #     split_date=split_dt,
    #     top_n_individual=TOP_N_INDIVIDUAL,
    #     title="Individual wallet cumulative PnL (train + test, resets at split)",
    # )
    # fig_ind.show(renderer="browser")

    del df_fills
else:
    print("PLOT_WALLET_PNL=False — skipping plots")
